In [ ]:
import kagglehub
import kagglehub
import kagglehub
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
cars_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(cars_path)

print(f"Dataset shape: {df.shape}")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle missing values as needed.")
    else:
        print("\nNo missing values found.")

check_missing_values(df)

In [ ]:
# Task 2: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)


In [ ]:
# Task 3: Write your code here:

# Encode features and target using LabelEncoder
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df

In [ ]:
# Task 4: Write your code here:
# Standardize features using StandardScaler
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  ### DON'T SCALE THE TARGET
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df

In [ ]:
  # Task 5: Write your code here:
import seaborn as sns

def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    print(df[target_column].value_counts(normalize=True))
    sns.countplot(x=df[target_column])
    plt.title("Target Distribution")
    plt.show()

check_target_imbalance(df, "Target")


# target not imblance

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df["Target"].astype(float)

In [ ]:
!pip install catboost

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# trian the model
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

model.fit(X_train, y_train)
print("Model trained!")

#calculate MAE
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")
#k-fold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE the average:  ${mae_scores.mean():,.2f}")







In [ ]:
# Task 1: Write your code here:
# Feature importance
importance = pd.DataFrame({
    'feature': y,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: